In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import shutil

input_asd_dir = '/kaggle/input/autism/EEGDATA/RAW  DATA/ASD GROUP/'
working_asd_dir = '/kaggle/working/ASD_GROUP/'

# Create destination directory if not exists
os.makedirs(working_asd_dir, exist_ok=True)

# Copy all .vhdr, .eeg, .vmrk files to writable folder
for file in os.listdir(input_asd_dir):
    if file.endswith(('.vhdr', '.eeg', '.vmrk')):
        shutil.copy2(os.path.join(input_asd_dir, file), os.path.join(working_asd_dir, file))

# Now update the .vhdr files in the writable folder
for filename in os.listdir(working_asd_dir):
    if filename.endswith('.vhdr'):
        vhdr_path = os.path.join(working_asd_dir, filename)
        base_name = filename[:-5]
        with open(vhdr_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
        for i, line in enumerate(lines):
            if line.startswith('DataFile='):
                lines[i] = f'DataFile={base_name}.eeg\n'
            if line.startswith('MarkerFile='):
                lines[i] = f'MarkerFile={base_name}.vmrk\n'
        with open(vhdr_path, 'w', encoding='utf-8') as file:
            file.writelines(lines)

print("Files copied and .vhdr updated in writable directory.")

Files copied and .vhdr updated in writable directory.


In [3]:
import mne
import numpy as np

# --- Configuration ---
# CHANGE THIS LINE to point to the .vhdr file instead of the .eeg file.
file_path = '/kaggle/working/ASD_GROUP/001.vhdr' 

# --- Load the Data ---
# Now, this should successfully load the metadata.
try:
    raw = mne.io.read_raw_brainvision(file_path, preload=False, verbose=False)
except Exception as e:
    print(f"Error loading file: {e}")
    # This block won't be executed if the file path is correct.
    exit()

# --- Extract and Print Information ---
print("="*40)
print(f"ANALYZING FILE: {file_path.split('/')[-1]}")
print("="*40)

# This part of the code will now work because 'raw' was successfully created.
n_channels = raw.info['nchan']
n_times = raw.n_times
print(f"🧠 Data Shape: {n_channels} channels, {n_times} time points")

sfreq = raw.info['sfreq']
duration_seconds = n_times / sfreq
duration_minutes = duration_seconds / 60
print(f"🕒 Sampling Frequency: {sfreq} Hz")
print(f"⏳ Session Duration: {duration_seconds:.2f} seconds (~{duration_minutes:.2f} minutes)")

print("\n" + "-"*40)
print("FULL METADATA OVERVIEW (`raw.info`):")
print("-" * 40)
print(raw.info)
print("="*40)

/tmp/ipykernel_75/2185931088.py:11: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(file_path, preload=False, verbose=False)


ANALYZING FILE: 001.vhdr
🧠 Data Shape: 64 channels, 1277960 time points
🕒 Sampling Frequency: 1000.0 Hz
⏳ Session Duration: 1277.96 seconds (~21.30 minutes)

----------------------------------------
FULL METADATA OVERVIEW (`raw.info`):
----------------------------------------
<Info | 7 non-empty values
 bads: []
 ch_names: Fp1, Fp2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T7, T8, P7, ...
 chs: 64 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 100.0 Hz
 meas_date: 2016-03-09 16:54:23 UTC
 nchan: 64
 projs: []
 sfreq: 1000.0 Hz
>


In [3]:
!pip install mne-icalabel
!pip install autoreject
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 66.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.0 MB/s eta 0:00:00


In [ ]:
import os
import mne
import numpy as np
import pywt
from mne.preprocessing import ICA
from mne_icalabel import label_components
from autoreject import AutoReject
import gc

# --- 1. SETUP FILE PATHS ---
asd_dir = '/kaggle/working/ASD_GROUP'
td_dir = '/kaggle/working/TD_GROUP'
subject_files = {
    'ASD': [os.path.join(asd_dir, f) for f in os.listdir(asd_dir) if f.endswith('.vhdr')],
    'TD': [os.path.join(td_dir, f) for f in os.listdir(td_dir) if f.endswith('.vhdr')]
}
base_save_dir = '/kaggle/working/preprocessed_data'
os.makedirs(base_save_dir, exist_ok=True)


# --- 2. HELPER FUNCTION (Wavelet Denoising) ---
def wavelet_denoise(signal, wavelet='db4', level=4, threshold_scale=0.7):
    coeffs = pywt.wavedec(signal, wavelet, level=level, mode='per')
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = threshold_scale * sigma * np.sqrt(2 * np.log(len(signal)))
    coeffs[1:] = [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
    denoised = pywt.waverec(coeffs, wavelet, mode='per')
    if len(denoised) > len(signal):
        denoised = denoised[:len(signal)]
    elif len(denoised) < len(signal):
        denoised = np.pad(denoised, (0, len(signal) - len(denoised)), mode='constant')
    return denoised


# --- 3. MAIN PREPROCESSING FUNCTION (HYBRID ICA-WAVELET METHOD) ---
def preprocess_subject(file_path, save_dir):
    subject_id = os.path.basename(file_path).split('.')[0]
    
    try:
        print(f"Processing {subject_id}...")
        raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)

        if 'EOG' in raw.ch_names:
            raw.set_channel_types({'EOG': 'eog'})
        raw.pick_types(eeg=True)
        
        raw.set_montage('standard_1020', on_missing='warn')
        raw.resample(250, npad='auto')
        raw.filter(l_freq=0.5, h_freq=100., fir_design='firwin', verbose=False)
        raw.set_eeg_reference('average', projection=True, verbose=False)
        raw.apply_proj()

        # --- STEP 1: REMOVE MAJOR ARTIFACTS WITH ICA ---
        ica = ICA(n_components=None, method='infomax', fit_params={'extended': True}, random_state=42)
        ica.fit(raw)
        
        ic_labels = label_components(raw, ica, method='iclabel')
        labels = ic_labels['labels'][:ica.n_components_]
        
        artifact_classes = ['eye blink', 'muscle artifact', 'channel noise']
        artifact_idx = [i for i, lab in enumerate(labels) if lab in artifact_classes]
        
        print(f"  -> Step 1: Found {len(artifact_idx)} major artifact components. Removing them.")
        ica.apply(raw, exclude=artifact_idx) # Use the stable, built-in removal method

        # --- STEP 2: DENOISE REMAINING SIGNAL WITH WAVELETS ---
        print(f"  -> Step 2: Applying wavelet denoising to each of the {len(raw.ch_names)} EEG channels.")
        # Get the data after ICA cleaning
        data_after_ica = raw.get_data()
        # Create a new array to hold the final denoised data
        final_denoised_data = np.zeros_like(data_after_ica)
        
        # Loop through each channel and apply the wavelet denoise function
        for i in range(data_after_ica.shape[0]):
            final_denoised_data[i, :] = wavelet_denoise(data_after_ica[i, :])
            
        # Put the fully cleaned data back into the raw object
        raw._data = final_denoised_data

        # --- Continue with the rest of the pipeline ---
        epochs = mne.make_fixed_length_epochs(raw, duration=4.0, overlap=1.5, preload=True, verbose=False)
        if len(epochs) == 0:
            print(f"⚠️ Could not create any epochs for {subject_id}.")
            return

        print(f"  -> Applying AutoReject to {len(epochs)} initial epochs.")
        ar = AutoReject(n_interpolate=[1, 2, 4], random_state=42, n_jobs=-1, verbose=False)
        epochs_clean = ar.fit_transform(epochs)
        
        epochs_fname = os.path.join(save_dir, f"{subject_id}_epochs-epo.fif")
        epochs_clean.save(epochs_fname, overwrite=True, verbose=False)
        print(f"✅ Saved {len(epochs_clean)} final cleaned epochs for {subject_id}")

    except Exception as e:
        print(f"❌ FAILED to process {subject_id}. Error: {e}")
    finally:
        if 'raw' in locals(): del raw
        if 'ica' in locals(): del ica
        if 'epochs' in locals(): del epochs
        if 'epochs_clean' in locals(): del epochs_clean
        gc.collect()


# --- 4. RUN THE ENTIRE PIPELINE ---
def run_pipeline(subject_files_dict, output_dir):
    for group, files in subject_files_dict.items():
        print(f"\n{'='*20} Processing Group: {group} {'='*20}")
        group_save_dir = os.path.join(output_dir, group)
        os.makedirs(group_save_dir, exist_ok=True)
        print(f"  -> Output will be saved in: {group_save_dir}\n")
        for file_path in files:
            preprocess_subject(file_path, group_save_dir)
    print("\n--- All processing complete. ---")

# --- 5. EXECUTE THE PIPELINE ---
run_pipeline(subject_files, base_save_dir)


==================== Processing Group: ASD ====================
  -> Output will be saved in: /kaggle/working/preprocessed_data/ASD

Processing 014...


/tmp/ipykernel_75/2312456318.py:41: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Fitting ICA to data using 63 channels (please be patient, this may take a while)
    Applying projection operator with 1 vector (pre-whitener computation)
    Applying projection operator with 1 vector (pre-whitener application)
Selecting by non-zero PCA components: 62 components
Computing Extended Infomax ICA
    Applying projection operator with 1 vector (pre-whitener application)
Fitting ICA took 116.5s.


/tmp/ipykernel_75/2312456318.py:57: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_75/2312456318.py:57: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


  -> Step 1: Found 17 major artifact components. Removing them.
Applying ICA to Raw instance
    Applying projection operator with 1 vector (pre-whitener application)
    Transforming to ICA space (62 components)
    Zeroing out 17 ICA components
    Projecting back using 63 PCA components
  -> Step 2: Applying wavelet denoising to each of the 63 EEG channels.
  -> Applying AutoReject to 494 initial epochs.
Dropped 9 epochs: 21, 75, 76, 77, 81, 193, 200, 456, 474
✅ Saved 485 final cleaned epochs for 014
Processing 001...


...

  -> Step 1: Found 21 major artifact components. Removing them.
Applying ICA to Raw instance
    Applying projection operator with 1 vector (pre-whitener application)
    Transforming to ICA space (62 components)
    Zeroing out 21 ICA components
    Projecting back using 63 PCA components
  -> Step 2: Applying wavelet denoising to each of the 63 EEG channels.
  -> Applying AutoReject to 629 initial epochs.
Dropped 20 epochs: 8, 13, 14, 19, 45, 46, 58, 60, 61, 139, 151, 157, 318, 331, 332, 345, 455, 563, 595, 628
✅ Saved 609 final cleaned epochs for 015
Processing 019...


/tmp/ipykernel_75/2312456318.py:41: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Fitting ICA to data using 63 channels (please be patient, this may take a while)
    Applying projection operator with 1 vector (pre-whitener computation)
    Applying projection operator with 1 vector (pre-whitener application)
Selecting by non-zero PCA components: 62 components
Computing Extended Infomax ICA
    Applying projection operator with 1 vector (pre-whitener application)
Fitting ICA took 186.5s.


/tmp/ipykernel_75/2312456318.py:57: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_75/2312456318.py:57: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


  -> Step 1: Found 24 major artifact components. Removing them.
Applying ICA to Raw instance
    Applying projection operator with 1 vector (pre-whitener application)
    Transforming to ICA space (62 components)
    Zeroing out 24 ICA components
    Projecting back using 63 PCA components
  -> Step 2: Applying wavelet denoising to each of the 63 EEG channels.
  -> Applying AutoReject to 637 initial epochs.


In [4]:
import os
import mne
import numpy as np
import pywt
from mne.preprocessing import ICA
from mne_icalabel import label_components
from autoreject import AutoReject
import gc

# --- 1. SETUP FILE PATHS ---
asd_dir = '/kaggle/working/ASD_GROUP'
td_dir = '/kaggle/working/TD_GROUP'
subject_files = {
    'ASD': [os.path.join(asd_dir, f) for f in os.listdir(asd_dir) if f.endswith('.vhdr')],
    'TD': [os.path.join(td_dir, f) for f in os.listdir(td_dir) if f.endswith('.vhdr')]
}
base_save_dir = '/kaggle/working/preprocessed_data'
os.makedirs(base_save_dir, exist_ok=True)


# --- 2. HELPER FUNCTION (Wavelet Denoising) ---
def wavelet_denoise(signal, wavelet='db4', level=4, threshold_scale=0.7):
    coeffs = pywt.wavedec(signal, wavelet, level=level, mode='per')
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    uthresh = threshold_scale * sigma * np.sqrt(2 * np.log(len(signal)))
    coeffs[1:] = [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
    denoised = pywt.waverec(coeffs, wavelet, mode='per')
    if len(denoised) > len(signal):
        denoised = denoised[:len(signal)]
    elif len(denoised) < len(signal):
        denoised = np.pad(denoised, (0, len(signal) - len(denoised)), mode='constant')
    return denoised


# --- 3. MAIN PREPROCESSING FUNCTION (HYBRID ICA-WAVELET METHOD) ---
# This function does not need to be changed.
def preprocess_subject(file_path, save_dir):
    subject_id = os.path.basename(file_path).split('.')[0]
    
    try:
        print(f"Processing {subject_id}...")
        raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)

        if 'EOG' in raw.ch_names:
            raw.set_channel_types({'EOG': 'eog'})
        raw.pick_types(eeg=True)
        
        raw.set_montage('standard_1020', on_missing='warn')
        raw.resample(250, npad='auto')
        raw.filter(l_freq=0.5, h_freq=100., fir_design='firwin', verbose=False)
        raw.set_eeg_reference('average', projection=True, verbose=False)
        raw.apply_proj()

        ica = ICA(n_components=None, method='infomax', fit_params={'extended': True}, random_state=42)
        ica.fit(raw)
        
        ic_labels = label_components(raw, ica, method='iclabel')
        labels = ic_labels['labels'][:ica.n_components_]
        
        artifact_classes = ['eye blink', 'muscle artifact', 'channel noise']
        artifact_idx = [i for i, lab in enumerate(labels) if lab in artifact_classes]
        
        print(f"  -> Step 1: Found {len(artifact_idx)} major artifact components. Removing them.")
        ica.apply(raw, exclude=artifact_idx)

        print(f"  -> Step 2: Applying wavelet denoising to each of the {len(raw.ch_names)} EEG channels.")
        data_after_ica = raw.get_data()
        final_denoised_data = np.zeros_like(data_after_ica)
        
        for i in range(data_after_ica.shape[0]):
            final_denoised_data[i, :] = wavelet_denoise(data_after_ica[i, :])
            
        raw._data = final_denoised_data

        epochs = mne.make_fixed_length_epochs(raw, duration=4.0, overlap=1.5, preload=True, verbose=False)
        if len(epochs) == 0:
            print(f"⚠️ Could not create any epochs for {subject_id}.")
            return

        print(f"  -> Applying AutoReject to {len(epochs)} initial epochs.")
        ar = AutoReject(n_interpolate=[1, 2, 4], random_state=42, n_jobs=-1, verbose=False)
        epochs_clean = ar.fit_transform(epochs)
        
        epochs_fname = os.path.join(save_dir, f"{subject_id}_epochs-epo.fif")
        epochs_clean.save(epochs_fname, overwrite=True, verbose=False)
        print(f"✅ Saved {len(epochs_clean)} final cleaned epochs for {subject_id}")

    except Exception as e:
        print(f"❌ FAILED to process {subject_id}. Error: {e}")
    finally:
        if 'raw' in locals(): del raw
        if 'ica' in locals(): del ica
        if 'epochs' in locals(): del epochs
        if 'epochs_clean' in locals(): del epochs_clean
        gc.collect()


# --- 4. MODIFIED RUN FUNCTION TO PROCESS ONLY REMAINING TD FILES ---
def run_pipeline(subject_files_dict, output_dir):
    
    # 📝 Manually define the list of TD subjects to skip
    td_subjects_to_skip = [
        '001', '002', '003', '004', '005', '006', '007', '008', 
        '011', '014', '016', '017'
    ]

    for group, files in subject_files_dict.items():
        
        # Skip the ASD group entirely
        if group == 'ASD':
            print(f"\n{'='*20} Skipping Group: {group} (Already Processed) {'='*20}")
            continue

        print(f"\n{'='*20} Processing Group: {group} {'='*20}")
        group_save_dir = os.path.join(output_dir, group)
        os.makedirs(group_save_dir, exist_ok=True)
        print(f"  -> Output will be saved in: {group_save_dir}\n")
        
        for file_path in files:
            subject_id = os.path.basename(file_path).split('.')[0]
            
            # Check if the current subject is in our manual skip list
            if subject_id in td_subjects_to_skip:
                print(f"-> Skipping {subject_id} (already processed).")
                continue # Move to the next file
            
            # If not in the skip list, process the subject
            preprocess_subject(file_path, group_save_dir)
            
    print("\n--- All processing complete. ---")

# --- 5. EXECUTE THE PIPELINE ---
run_pipeline(subject_files, base_save_dir)


==================== Skipping Group: ASD (Already Processed) ====================

==================== Processing Group: TD ====================
  -> Output will be saved in: /kaggle/working/preprocessed_data/TD

-> Skipping 003 (already processed).
Processing 013...


/tmp/ipykernel_37/1207062191.py:42: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Fitting ICA to data using 63 channels (please be patient, this may take a while)
    Applying projection operator with 1 vector (pre-whitener computation)
    Applying projection operator with 1 vector (pre-whitener application)
Selecting by non-zero PCA components: 62 components
Computing Extended Infomax ICA
    Applying projection operator with 1 vector (pre-whitener application)
Fitting ICA took 124.1s.


/tmp/ipykernel_37/1207062191.py:57: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_37/1207062191.py:57: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


  -> Step 1: Found 17 major artifact components. Removing them.
Applying ICA to Raw instance
    Applying projection operator with 1 vector (pre-whitener application)
    Transforming to ICA space (62 components)
    Zeroing out 17 ICA components
    Projecting back using 63 PCA components
  -> Step 2: Applying wavelet denoising to each of the 63 EEG channels.
  -> Applying AutoReject to 688 initial epochs.
Dropped 67 epochs: 1, 2, 9, 42, 43, 44, 45, 50, 52, 88, 89, 119, 205, 219, 342, 343, 344, 374, 375, 378, 384, 385, 395, 396, 397, 398, 399, 407, 408, 421, 422, 423, 460, 461, 462, 464, 465, 466, 467, 468, 469, 473, 474, 483, 484, 485, 507, 508, 549, 588, 589, 590, 591, 592, 594, 595, 596, 597, 598, 623, 624, 625, 626, 627, 640, 641, 672
✅ Saved 621 final cleaned epochs for 013
Processing 009...


...

  -> Step 1: Found 18 major artifact components. Removing them.
Applying ICA to Raw instance
    Applying projection operator with 1 vector (pre-whitener application)
    Transforming to ICA space (62 components)
    Zeroing out 18 ICA components
    Projecting back using 63 PCA components
  -> Step 2: Applying wavelet denoising to each of the 63 EEG channels.
  -> Applying AutoReject to 531 initial epochs.
Dropped 59 epochs: 4, 5, 6, 9, 19, 20, 27, 28, 37, 56, 66, 71, 79, 80, 94, 126, 155, 159, 173, 175, 192, 193, 201, 216, 217, 225, 240, 241, 244, 245, 248, 249, 285, 304, 315, 325, 326, 331, 397, 398, 416, 427, 440, 441, 446, 448, 451, 459, 460, 465, 466, 489, 494, 514, 515, 520, 521, 525, 526
✅ Saved 472 final cleaned epochs for 018
-> Skipping 017 (already processed).
Processing 019...


/tmp/ipykernel_37/1207062191.py:42: RuntimeWarning: Online software filter detected. Using software filter settings and ignoring hardware values
  raw = mne.io.read_raw_brainvision(file_path, preload=True, verbose=False)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...
Fitting ICA to data using 63 channels (please be patient, this may take a while)
    Applying projection operator with 1 vector (pre-whitener computation)
    Applying projection operator with 1 vector (pre-whitener application)
Selecting by non-zero PCA components: 62 components
Computing Extended Infomax ICA
    Applying projection operator with 1 vector (pre-whitener application)
Fitting ICA took 175.2s.


/tmp/ipykernel_37/1207062191.py:57: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')
/tmp/ipykernel_37/1207062191.py:57: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


  -> Step 1: Found 24 major artifact components. Removing them.
Applying ICA to Raw instance
    Applying projection operator with 1 vector (pre-whitener application)
    Transforming to ICA space (62 components)
    Zeroing out 24 ICA components
    Projecting back using 63 PCA components
  -> Step 2: Applying wavelet denoising to each of the 63 EEG channels.
  -> Applying AutoReject to 637 initial epochs.
Dropped 46 epochs: 4, 5, 9, 26, 76, 87, 90, 91, 93, 94, 131, 133, 134, 137, 139, 140, 145, 179, 185, 187, 188, 205, 206, 234, 333, 334, 361, 362, 402, 404, 407, 412, 413, 448, 449, 469, 518, 519, 529, 580, 595, 596, 597, 607, 612, 616
✅ Saved 591 final cleaned epochs for 019

--- All processing complete. ---
